In [4]:
import os
from typing import Optional
from urllib.parse import urljoin
import pandas as pd
from bs4 import BeautifulSoup
# Cache để lưu trữ kết quả đã trích xuất

DATA_HTML_DIR = "./data"

def extract_image_from_html(doc_id: str, base_url: str = "") -> Optional[str]:
    """
    Trích xuất ảnh từ file HTML gốc
    Ưu tiên: Open Graph image > Twitter Card > First img tag > Favicon
    """
    global images_cache
    
    # Kiểm tra cache
    if doc_id in images_cache:
        return images_cache[doc_id]
    
    try:
        html_path = os.path.join(DATA_HTML_DIR, f"{doc_id}.html")
        
        if not os.path.exists(html_path):
            print(f"HTML file not found: {html_path}")
            images_cache[doc_id] = None
            return None
        
        with open(html_path, 'r', encoding='utf-8') as f:
            html_content = f.read()
        
        soup = BeautifulSoup(html_content, 'html.parser')
        
        # 1. Tìm Open Graph image (og:image)
        og_image = soup.find('meta', property='og:image')
        if og_image and og_image.get('content'):
            image_url = og_image['content']
            if image_url.startswith('http'):
                images_cache[doc_id] = image_url
                return image_url
            elif base_url:
                full_url = urljoin(base_url, image_url)
                images_cache[doc_id] = full_url
                return full_url
        
        # 2. Tìm Twitter Card image
        twitter_image = soup.find('meta', attrs={'name': 'twitter:image'})
        if twitter_image and twitter_image.get('content'):
            image_url = twitter_image['content']
            if image_url.startswith('http'):
                images_cache[doc_id] = image_url
                return image_url
            elif base_url:
                full_url = urljoin(base_url, image_url)
                images_cache[doc_id] = full_url
                return full_url
        
        # 3. Tìm ảnh đầu tiên trong <head> hoặc <body>
        # Ưu tiên ảnh có kích thước lớn
        images = soup.find_all('img', src=True)
        for img in images:
            src = img.get('src', '')
            if src:
                # Bỏ qua các ảnh quá nhỏ hoặc icon
                width = img.get('width', '')
                height = img.get('height', '')
                
                # Kiểm tra xem có phải ảnh lớn không
                try:
                    if width and int(width) < 100:
                        continue
                    if height and int(height) < 100:
                        continue
                except:
                    pass
                
                # Bỏ qua các file icon
                if any(keyword in src.lower() for keyword in ['icon', 'logo', 'favicon', 'sprite']):
                    continue
                
                if src.startswith('http'):
                    images_cache[doc_id] = src
                    return src
                elif base_url:
                    full_url = urljoin(base_url, src)
                    images_cache[doc_id] = full_url
                    return full_url
        
        # 4. Tìm favicon nếu không có ảnh nào khác
        favicon = soup.find('link', rel=lambda x: x and 'icon' in x.lower())
        if favicon and favicon.get('href'):
            favicon_url = favicon['href']
            if favicon_url.startswith('http'):
                images_cache[doc_id] = favicon_url
                return favicon_url
            elif base_url:
                full_url = urljoin(base_url, favicon_url)
                images_cache[doc_id] = full_url
                return full_url
        
        # Không tìm thấy ảnh nào
        images_cache[doc_id] = None
        return None
        
    except Exception as e:
        print(f"Error extracting image from HTML {doc_id}: {e}")
        images_cache[doc_id] = None
        return None

In [5]:
import pandas as pd

# Cache ảnh
images_cache = {}

# Đường dẫn file
INPUT_CSV = "./final_document_tfidf_pagerank.csv"
OUTPUT_CSV = "docs_with_images.csv"

# Đọc file CSV
df = pd.read_csv(INPUT_CSV)

# Kiểm tra cột doc_id
if "document" in df.columns:
    # document đang là dạng "0.txt" → lấy số làm doc_id
    df["doc_id"] = df["document"].str.replace(".txt", "", regex=False)
elif "id" in df.columns:
    df["doc_id"] = df["id"].astype(str)
else:
    raise ValueError("Không tìm thấy cột id hoặc document")

# Lấy base_url từ cột url
def get_image(row):
    doc_id = row["doc_id"]
    base_url = row["url"]
    return extract_image_from_html(doc_id, base_url)

# Chạy trích xuất ảnh
df["image_url"] = df.apply(get_image, axis=1)

# Tạo dataframe mới chỉ gồm doc_id và image_url
output_df = df[["doc_id", "image_url"]]

# Lưu ra CSV mới
output_df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")

print(f"Đã lưu file: {OUTPUT_CSV}")


Đã lưu file: docs_with_images.csv
